# UrbanCart E-Commerce Analytics — Python/Pandas AnalysisData cleaning, outlier detection, and hypothesis-driven exploratory analysis testing five competing explanations for a stalled revenue growth pattern.

## 1. Setup

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltcustomers = pd.read_csv('customers.csv', na_values=['\\N'])products = pd.read_csv('products.csv', na_values=['\\N'])orders = pd.read_csv('orders.csv', na_values=['\\N'])q_items = pd.read_csv('order_items.csv', na_values=['\\N'])deliveries = pd.read_csv('deliveries.csv', na_values=['\\N'])# na_values=['\\N'] tells Pandas to treat MySQL's NULL marker (\\N) as a true missing value,# instead of reading it as the literal text string "\\N".q_items.head()

## 2. Data Cleaning### 2.1 Missing value handling

In [ ]:
# quantity: impute with median (not mode) — distribution isn't skewed enough to# justify filling every missing value with the mode (1); median (2) is the safer,# less-biased choice since it avoids systematically understating revenue.print(q_items['quantity'].describe())median_qty = q_items['quantity'].median()q_items['quantity'] = q_items['quantity'].fillna(median_qty)print("Remaining missing quantity:", q_items['quantity'].isna().sum())

In [ ]:
# city / customer_segment: fill with "Unknown" rather than dropping rows or leaving NULL,# so these customers stay visible as their own bucket in any regional/segment grouping.print("Missing city before:", customers['city'].isna().sum())print("Missing segment before:", customers['customer_segment'].isna().sum())customers['city'] = customers['city'].fillna('Unknown')customers['customer_segment'] = customers['customer_segment'].fillna('Unknown')print("Missing city after:", customers['city'].isna().sum())print("Missing segment after:", customers['customer_segment'].isna().sum())

### 2.2 Outlier detection (IQR method) on `unit_price`

In [ ]:
Q1 = q_items['unit_price'].quantile(0.25)Q3 = q_items['unit_price'].quantile(0.75)IQR = Q3 - Q1upper_bound = Q3 + 1.5 * IQRoutliers = q_items[q_items['unit_price'] > upper_bound]print(f"Number of outliers: {len(outliers)}")outliers.sort_values('unit_price', ascending=False).head()

In [ ]:
# Validate outliers aren't just legitimately expensive products —# compare each flagged price to that product's typical (clean) price.clean_prices = q_items[~q_items['unit_price'].isin(outliers['unit_price'])]avg_price_clean = clean_prices.groupby('product_id')['unit_price'].mean().reset_index()avg_price_clean.columns = ['product_id', 'clean_avg_price']check = outliers.merge(avg_price_clean, on='product_id')check['price_ratio'] = check['unit_price'] / check['clean_avg_price']check.sort_values('price_ratio', ascending=False)[['product_id','unit_price','clean_avg_price','price_ratio']].head(10)# Ratios cluster ~7x-11x across all 133 flagged rows — confirms data-entry errors, not real premium pricing.

In [ ]:
# Fix: replace each outlier's price with that product's clean average priceq_items = q_items.merge(avg_price_clean, on='product_id', how='left')outlier_idx = outliers.indexq_items.loc[outlier_idx, 'unit_price'] = q_items.loc[outlier_idx, 'clean_avg_price']q_items = q_items.drop(columns=['clean_avg_price'])print("Max unit_price after fix:", q_items['unit_price'].max())print("Remaining rows above old upper_bound:", (q_items['unit_price'] > upper_bound).sum())

### 2.3 Category standardization(City standardization was applied in MySQL; category is reapplied here since Python reads from the original CSV.)

In [ ]:
products['category'] = products['category'].replace({    'fashion': 'Fashion',    'Home and Kitchen': 'Home & Kitchen',    'home & kitchen': 'Home & Kitchen',    'ELECTRONICS': 'Electronics',    'electronics': 'Electronics'})customers['city'] = customers['city'].replace({    'hyderabad': 'Hyderabad', 'HYDERABAD': 'Hyderabad', 'Hyd': 'Hyderabad',    'mumbai': 'Mumbai', 'Bombay': 'Mumbai',    'bangalore': 'Bangalore', 'Bengaluru': 'Bangalore',    'delhi': 'Delhi', 'New Delhi': 'Delhi',    'chennai': 'Chennai', 'Madras': 'Chennai',    'pune': 'Pune',    'kolkata': 'Kolkata', 'Calcutta': 'Kolkata',    'ahmedabad': 'Ahmedabad'})print(products['category'].unique())print(customers['city'].unique())

### 2.4 Referential integrity check

In [ ]:
orphan_orders = orders[~orders['customer_id'].isin(customers['customer_id'])]orphan_items = q_items[~q_items['product_id'].isin(products['product_id'])]print("Orphan orders (bad customer_id):", len(orphan_orders))print("Orphan order_items (bad product_id):", len(orphan_items))# Both 0 — confirms MySQL's foreign key constraints held throughout loading and cleaning.

### 2.5 Save cleaned data

In [ ]:
q_items['revenue'] = q_items['quantity'] * q_items['unit_price']customers.to_csv('customers_clean.csv', index=False)products.to_csv('products_clean.csv', index=False)orders.to_csv('orders_clean.csv', index=False)q_items.to_csv('order_items_clean.csv', index=False)print("All cleaned files saved successfully.")

## 3. Hypothesis-Driven EDAThe business problem: revenue growth has stalled over the last two quarters. Five hypotheses are tested against the cleaned data below.

### 3.1 Revenue by category

In [ ]:
q_items_full = q_items.merge(products[['product_id', 'category']], on='product_id', how='left')category_revenue = q_items_full.groupby('category')['revenue'].sum().sort_values(ascending=False)category_revenue

### 3.2 Monthly revenue trend

In [ ]:
orders_items_full = q_items.merge(orders[['order_id', 'order_date']], on='order_id', how='left')orders_items_full['order_date'] = pd.to_datetime(orders_items_full['order_date'])orders_items_full['order_month'] = orders_items_full['order_date'].dt.to_period('M')monthly_revenue = orders_items_full.groupby('order_month')['revenue'].sum()monthly_revenue

In [ ]:
monthly_revenue.plot(kind='line', marker='o', figsize=(10,5), title='Monthly Revenue Trend')plt.ylabel('Revenue (₹)')plt.xlabel('Month')plt.tight_layout()plt.show()# Revenue oscillates between ~₹9.6M-11.9M with no growth trend — confirms the "stalled growth" problem precisely.

### 3.3 Hypothesis 1 — Is new customer acquisition declining?

In [ ]:
customers['signup_date'] = pd.to_datetime(customers['signup_date'])customers['signup_month'] = customers['signup_date'].dt.to_period('M')new_customers_monthly = customers.groupby('signup_month')['customer_id'].count()new_customers_monthly# Steady 173-226 signups/month — hypothesis rejected.

### 3.4 Hypothesis 2 — Are customers not repeat-buying?

In [ ]:
orders_per_customer = orders.groupby('customer_id')['order_id'].count()repeat_customers = (orders_per_customer > 1).sum()total_customers = orders_per_customer.count()repeat_rate = repeat_customers / total_customers * 100print(f"Total customers with orders: {total_customers}")print(f"Repeat customers: {repeat_customers}")print(f"Repeat purchase rate: {repeat_rate:.2f}%")# 84.41% repeat rate — hypothesis rejected.

### 3.5 Hypothesis 3 — Is Average Order Value shrinking?

In [ ]:
order_value = orders_items_full.groupby('order_id')['revenue'].sum().reset_index()order_value = order_value.merge(orders[['order_id','order_date']], on='order_id')order_value['order_date'] = pd.to_datetime(order_value['order_date'])order_value['order_month'] = order_value['order_date'].dt.to_period('M')aov_monthly = order_value.groupby('order_month')['revenue'].mean()aov_monthly# Flat, oscillating ₹10,850-12,230 — hypothesis rejected.

### 3.6 Hypothesis 4 — Is one region underperforming?

In [ ]:
step1 = q_items.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')step2 = step1.merge(customers[['customer_id', 'city']], on='customer_id', how='left')city_revenue = step2.groupby('city')['revenue'].sum().sort_values(ascending=False)city_revenue# Moderate 25-30% spread (Chennai highest, Pune lowest) — no single region is dragging down the aggregate.

### 3.7 Revenue concentration — is revenue driven by a small customer segment?

In [ ]:
customer_revenue = step2.groupby('customer_id')['revenue'].sum().sort_values(ascending=False)customer_revenue.describe()

In [ ]:
top_10pct_count = int(len(customer_revenue) * 0.10)top_10pct_revenue = customer_revenue.head(top_10pct_count).sum()total_revenue = customer_revenue.sum()print(f"Top 10% of customers ({top_10pct_count} customers) generate: {top_10pct_revenue:,.2f}")print(f"Total revenue: {total_revenue:,.2f}")print(f"% of revenue from top 10%: {top_10pct_revenue/total_revenue*100:.2f}%")# 32.65% — confirmed. This is the one finding that holds up: revenue is meaningfully# concentrated among a relatively small group of high-value repeat customers.

### 3.8 Hypothesis 5 — Do delivery delays hurt retention?

In [ ]:
orders_delivery = orders.merge(deliveries[['order_id', 'delivery_status']], on='order_id', how='left')orders_delivery = orders_delivery.sort_values(['customer_id', 'order_date'])orders_delivery['order_rank'] = orders_delivery.groupby('customer_id').cumcount() + 1first_orders = orders_delivery[orders_delivery['order_rank'] == 1]delayed_first = first_orders[first_orders['delivery_status'] == 'Delayed']['customer_id']ontime_first = first_orders[first_orders['delivery_status'] == 'On-time']['customer_id']total_orders_per_cust = orders.groupby('customer_id')['order_id'].count()delayed_repeat_rate = (total_orders_per_cust[total_orders_per_cust.index.isin(delayed_first)] > 1).mean() * 100ontime_repeat_rate = (total_orders_per_cust[total_orders_per_cust.index.isin(ontime_first)] > 1).mean() * 100print(f"Repeat rate when FIRST order was delayed: {delayed_repeat_rate:.2f}%")print(f"Repeat rate when FIRST order was on-time: {ontime_repeat_rate:.2f}%")# Only a ~2.6 point gap — hypothesis rejected as a material driver.

## 4. Summary| Hypothesis | Result ||---|---|| Fewer new customers | Rejected — steady signups || Customers not repeat-buying | Rejected — 84.41% repeat rate || AOV shrinking | Rejected — flat AOV || One region underperforming | Rejected — moderate, even spread || Delivery delays hurting retention | Rejected — only ~2.6pt gap || **Revenue concentrated in top customers** | **Confirmed — top 10% generate 32.65% of revenue** |Every individually-tested lever (acquisition, retention, AOV, region, delivery experience) is healthy. The one real, actionable finding is that revenue is disproportionately concentrated among a relatively small set of high-value repeat customers — directly informing the retention/VIP-program recommendation in the business write-up.